In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/aieducation/what-course-are-you-going-to-take/src/rewritten/o4_mini_high_small/checkpoints/post_cell_41_try_1.pickle

In [ ]:
%%cudf.pandas.profile
### cell 42 ###
# explicitly pick up all numeric columns instead of relying on numeric_only=True (which falls back to CPU in cuDF)
numeric_cols = wkrpt.select_dtypes(include=["number"])  # this is a cuDF‐aware pandas API
numeric_cols = numeric_cols.columns.tolist()
# drop the grouping key if it were numeric
def remove_group_key(cols, key="course_code"):  
    if key in cols:  
        cols.remove(key)  
    return cols
numeric_cols = remove_group_key(numeric_cols, "course_code")

wkrpt_mean = (
    wkrpt[["course_code"] + numeric_cols]
       .groupby("course_code")
       .mean()  # now fully on the GPU
       .sort_values("course_rating_int", ascending=False)
)
wkrpt_mean